# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This week runs one audit in two directions. First I read two findings from FlyRank's March 2026 research paper the way I want my own work read. Then I turn the same questions on my Week-5 model: an honest split, a leak hunt, and a rewrite of any claim that ran ahead of its evidence.

## 1. Two paper findings + my methodology questions

For each finding I answer four questions:

1. What does it claim?
2. Where does the outcome number come from?
3. Could the evaluation design have caught it being wrong?
4. Where does the claim stop being supported?

Some goodwill up front: the paper discloses its own limits ("No p-values or confidence intervals are reported", "Observational study: correlations do not prove causation"). My job is not to grade it. My job is to practice these questions before someone asks them of me.

---

**Finding A: the Freshness Multiplier (paper finding #4, tagged CONFIRMED).**

*The claim:* pages aged 365+ days that were refreshed within the last 30 days gained 3.2x in health score (10.7 to 34.5) and drew 57x more impressions (71 to 4,039). The paper calls refresh timing "one of the strongest measured levers available."

*Where the outcome comes from:* two numbers from the same snapshot. Impressions come from Google Search Console. Health score is FlyRank's own composite, built by a fixed recipe: 30 points impressions + 30 points position + 20 points CTR + 20 points scroll depth. Which pages count as refreshed comes from a days-since-update field.

*My questions:*

1. **Who gets refreshed?** Pages with proven visibility. The paper's own playbook says to start there. So the refreshed group selected itself, and no control group of comparable unrefreshed pages exists. Part of that gap was probably there before anyone refreshed anything.
2. **Are the two headline numbers two pieces of evidence?** No. Impressions sit inside the health score (30 of 100 points) and CTR is impressions-linked on both sides of its ratio. The 3.2x and the 57x share ingredients. They are related measurements, not two confirmations.
3. **How solid are the buckets?** Right next to the headline, the paper shows the 361-day-fresh bucket at 283 growing versus 1 declining and refuses to headline it. Good instinct. The 3.2x and 57x deserve the same caution, and no intervals are reported for them either.

*What I would ask the authors:* show refreshed versus unrefreshed pages matched on visibility floor and age, or publish the bucket sizes behind the multipliers. Either lets a reader separate refresh effect from selection.

---

**Finding B: the Content Performance Curve (paper finding #2, tagged CONFIRMED).**

*The claim:* health peaks at 61-90 days of age (33.1), decays to 14 by day 271-365, then rebounds to 25.1 after a year. Read as a lifecycle with a decay cliff.

*Where the outcome comes from:* health score again, bucketed by content age, measured once. Every age bucket is a different set of pages frozen at the same moment. Nothing is followed forward.

*My questions:*

1. **Can one snapshot tell a lifecycle story?** Only if the buckets behave like cohorts, and they cannot quite. Pages that survive to 365+ days are survivors. And the portfolio changed fast: the paper's own trend table grows active content from about 17K (Oct 2025) to 178K (Mar 2026), so young and old buckets lived through different conditions.
2. **Does a composite outcome support the word performance?** Health is FlyRank business logic, not an outside measure like clicks. A falling recipe sum is observed. Calling it performance decay adds a step the wording glides over.
3. **Credit where due:** the paper narrows its own rebound claim ("older pages can recover when they are updated well"). That is good claim hygiene, and it quietly concedes the 365+ point is refresh-selected, which loops straight back to Finding A.

*What I would ask the authors:* follow one publication month forward, month by month. If the cliff shows up there too, the lifecycle reading gets much stronger. If not, the curve is a snapshot artifact worth relabeling.

Both findings land close to home. My Week-5 model leans on the same age signal, and my label is also a proxy built from later data. Sections 2-4 turn the same questions on my own pipeline.

In [27]:
# Rebuilds the Week-5 development frame (same tables, filters, features, label) so every
# later section runs on identical data. The frame builder is Week 5's SQL parameterized by
# feature month; the label window's partitions are derived automatically because a January
# cutoff spills into March. Section 2 reuses it for earlier/later frames without copying.
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import json
from datetime import timedelta, date
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import duckdb

SEED = 42  # every stochastic step in this notebook uses this one seed

print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| duckdb", duckdb.__version__)

# --- token resolution: env var -> Colab secret -> repo .env -> interactive prompt ---
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN and Path("../../.env").exists():
    for line in Path("../../.env").read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")


def _months_between(start, end):
    """Calendar months (YYYY-MM) covering [start, end], inclusive."""
    months = []
    y, m = start.year, start.month
    while (y, m) <= (end.year, end.month):
        months.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13:
            y, m = y + 1, 1
    return months


def build_frame(feature_month: str):
    """Week-5 frame SQL, parameterized by FEATURE month. The label window runs
    cutoff+1 .. cutoff+30 and can spill past the next calendar month (a January
    cutoff reaches into March), so the builder derives every partition the label
    window needs instead of assuming a single future partition."""
    fact_feat = f"read_parquet('{REL}/fact_content_daily_performance/{feature_month}/*.parquet')"

    cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {fact_feat}").fetchone()[0]
    recent_start = cutoff_date - timedelta(days=29)
    label_start = cutoff_date + timedelta(days=1)
    label_end = cutoff_date + timedelta(days=30)

    fut_months = _months_between(label_start, label_end)
    fut_paths = ", ".join(
        f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in fut_months
    )
    fact_fut = f"read_parquet([{fut_paths}])"

    part_min, part_max = con.sql(
        f"SELECT MIN(report_date), MAX(report_date) FROM {fact_fut}"
    ).fetchone()
    print(f"Feature month {feature_month} | cutoff {cutoff_date} | "
          f"label window {label_start} .. {label_end} | "
          f"label partitions {fut_months} | bounds {part_min} .. {part_max}")
    assert str(part_min) == str(label_start), "Label partitions start late - label window uncovered"
    assert part_max >= label_end, "Label partitions end before the label window - label window uncovered"

    frame = con.sql(f"""
        WITH recent AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS recent30_impressions,
                SUM(gsc_clicks) AS recent30_clicks,
                AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
                COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
                COUNT(DISTINCT report_date) AS recent30_days
            FROM {fact_feat}
            WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        ),
        future AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS future30_impressions,
                COUNT(DISTINCT report_date) AS future30_days
            FROM {fact_fut}
            WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT
            r.client_hash_id,
            r.content_hash_id,
            r.recent30_impressions,
            LN(1 + r.recent30_impressions) AS log_recent30_impressions,
            100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
            r.recent30_avg_position,
            r.recent30_active_days,
            DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
            f.future30_impressions,
            f.future30_days,
            CASE
                WHEN r.recent30_impressions >= 100
                     AND f.future30_impressions < 0.80 * r.recent30_impressions
                THEN 1 ELSE 0
            END AS is_declining_next30
        FROM recent r
        INNER JOIN future f USING (client_hash_id, content_hash_id)
        LEFT JOIN (
            SELECT client_hash_id, content_hash_id, content_created_date
            FROM {DIM_CONTENT}
        ) c USING (client_hash_id, content_hash_id)
        WHERE r.recent30_days >= 14
          AND f.future30_days >= 14
          AND r.recent30_impressions >= 100
    """).df()

    assert not frame.duplicated(["client_hash_id", "content_hash_id"]).any(), \
        "Delivered frame is not one row per client-content grain"

    # Stable row order => deterministic ranking metrics (SQL GROUP BY order is not stable).
    frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

    meta = {
        "feature_month": feature_month,
        "future_partitions": fut_months,
        "cutoff": str(cutoff_date),
        "label_window": [str(label_start), str(label_end)],
        "rows": int(len(frame)),
        "base_rate": float(frame["is_declining_next30"].mean()),
        "clients": int(frame["client_hash_id"].nunique()),
        "largest_client_share": float(frame["client_hash_id"].value_counts(normalize=True).iloc[0]),
    }
    return frame, meta


df_march, meta_march = build_frame("month=2026-03")
df = df_march.copy()

print(f"\nFeature frame rows: {len(df):,}")
print(f"Label distribution: {df['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {meta_march['cutoff']}")

assert len(df) == 95810, (
    f"Frame size {len(df):,} != Week-4/5 receipt of 96,268 - investigate before continuing"
)
print("Frame-size receipt vs Weeks 4-5: PASS")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

TARGET = "is_declining_next30"
FEATURES = [
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
]

n_missing = int(df[FEATURES].isna().sum().sum())
print(f"\nMissing values across the 5 contracted features: {n_missing}")
df_model = df.dropna(subset=FEATURES).copy()
print(f"Rows available for modeling: {len(df_model):,}")


def make_baseline_scores(frame):
    """Verbatim re-implementation of the Week-4 hand rule. No fitted parameters."""
    is_visible = (frame["recent30_impressions"] >= 500).astype(int)
    is_top10 = ((frame["recent30_avg_position"] > 0) & (frame["recent30_avg_position"] <= 10)).astype(int)
    is_low_ctr = (frame["recent30_ctr_pct"] < 1.0).astype(int)
    is_stale = (frame["content_age_days"] >= 91).astype(int)
    low_ctr_top10 = is_visible * is_top10 * is_low_ctr
    visible_stale = is_visible * is_stale
    return 0.40 * is_visible + 0.35 * low_ctr_top10 + 0.25 * visible_stale


def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0


def recall_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    positive_total = labels.sum()
    if positive_total == 0:
        return 0.0
    return float(top_k.sum() / positive_total)


def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    gains = (2 ** labels[order[:k]] - 1) / np.log2(np.arange(2, k + 2))
    ideal = np.sort(labels)[::-1][:k]
    ideal_gains = (2 ** ideal - 1) / np.log2(np.arange(2, k + 2))
    denom = ideal_gains.sum()
    if denom == 0:
        return 0.0
    return float(gains.sum() / denom)


models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    ),
}

print("\nFrame, metric functions, hand rule and models loaded (all verbatim from Weeks 4-5).")
print("Two metrics join this week:")
print("- ROC-AUC: can the score tell declining from healthy pages at every threshold?")
print("  Week 5 only judged order near the top; this judges separation overall.")
print("- Brier score: mean squared error of predicted probability vs what happened.")
print("  Zero is perfect; a coin flip on this ~51/49 label lands near 0.25. Ranking")
print("  metrics cannot see calibration - two models can rank identically while one calls")
print("  everything 90% sure. The moment an editor cuts at 'show pages above 70%', the")
print("  probability values themselves become part of the claim.")
print("  Caveat: the hand rule emits scores, not probabilities, so its Brier row is rough.")


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
pandas 3.0.3 | numpy 2.5.1 | scikit-learn 1.9.0 | duckdb 1.5.4
Feature month month=2026-03 | cutoff 2026-03-31 | label window 2026-04-01 .. 2026-04-30 | label partitions ['2026-04'] | bounds 2026-04-01 .. 2026-04-30

Feature frame rows: 96,268
Label distribution: 51.1% declining
Cutoff date: 2026-03-31
Frame-size receipt vs Weeks 4-5: PASS

Missing values across the 5 contracted features: 0
Rows available for modeling: 96,268

Frame, metric functions, hand rule and models loaded (all verbatim from Weeks 4-5).
Two metrics join this week:
- ROC-AUC: can the score tell declining from healthy pages at every threshold?
  Week 5 only judged order near the top; this judges separation overall.
- Brier score: mean squared error of predicted probability vs what happened.
  Zero is perfect; a coin flip on this ~51/49 label lands near 0.25. Ranking
  metrics cannot see calibration - two models can rank identically while one calls
  

In [28]:
# Part 1 checks the paper's arithmetic against its own printed tables. Careful
# reading starts with the numbers that CAN be checked.
# Part 2 puts the paper's Finding B question to my own frame: decline rate by age bucket.
print("=== (a) The paper's claims vs its own tables ===")
checks = [
    ("Refresh health boost   34.5 / 10.7",                 3.2,   34.5 / 10.7),
    ("Refresh impressions    4039 / 71",                   57.0,  4039 / 71),
    ("361d-fresh bucket      283 growing : 1 declining",   283.0, 283 / 1),
]
arith_rows = []
for label, claimed, recomputed in checks:
    ok = abs(recomputed - claimed) / claimed < 0.05
    arith_rows.append({"claim": label, "paper_value": claimed,
                       "recomputed": round(recomputed, 2), "match_5pct": ok})
arith_table = pd.DataFrame(arith_rows)
print(arith_table.to_string(index=False))
assert arith_table["match_5pct"].all(), "Paper arithmetic did not reproduce - re-read before critiquing"

health_recipe = {"impressions": 30, "position": 30, "ctr": 20, "scroll_depth": 20}
print(f"\nHealth-score recipe (paper p.5/p.36): {health_recipe}")
print("Impressions enter it directly (30 of 100 points) and CTR (20 points) is")
print("impressions-linked on both sides of its ratio. Finding A's '3.2x health' and")
print("'57x impressions' therefore share ingredients: one measurement, two formulas.")

print("\n=== (b) The Finding B question on MY data (March frame) ===")
bins = [-1, 30, 90, 180, 365, 10 ** 9]
labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
age_table = (
    df_model.assign(age_bucket=pd.cut(df_model["content_age_days"], bins=bins, labels=labels))
            .groupby("age_bucket", observed=True)
            .agg(pages=(TARGET, "size"), decline_rate=(TARGET, "mean"))
)
print(age_table.round(3).to_string())

print("\nThe paper's health curve rises to 33.1 at 61-90d, falls to 14 at 271-365d, rebounds to 25.1.")
print("Mine answers the same lifecycle question with a different outcome (next-30-day decline")
print("probability) on a different population (my frame keeps only pages measurable next month;")
print("Section 3 counts them). Shape check, not replication:")
peak_bucket = age_table["decline_rate"].idxmax()
print(f"- My decline rate peaks at {peak_bucket}, then FALLS in the oldest bucket. Non-monotonic,")
print("  just like the paper's curve. Bucket shapes invite lifecycle readings that a single")
print("  cross-section cannot really support - in my data too.")

=== (a) The paper's claims vs its own tables ===
                                           claim  paper_value  recomputed  match_5pct
              Refresh health boost   34.5 / 10.7          3.2        3.22        True
                Refresh impressions    4039 / 71         57.0       56.89        True
361d-fresh bucket      283 growing : 1 declining        283.0      283.00        True

Health-score recipe (paper p.5/p.36): {'impressions': 30, 'position': 30, 'ctr': 20, 'scroll_depth': 20}
Impressions enter it directly (30 of 100 points) and CTR (20 points) is
impressions-linked on both sides of its ratio. Finding A's '3.2x health' and
'57x impressions' therefore share ingredients: one measurement, two formulas.

=== (b) The Finding B question on MY data (March frame) ===
            pages  decline_rate
age_bucket                     
0-30d        6721         0.277
31-90d      23724         0.467
91-180d     15777         0.615
181-365d    36369         0.549
365d+       13677    

## 2. My model under an honest split (before/after)

Week 5 already used a client-grouped split inside March. What it never tested was time. Four rows, one question each:

- **Row 1, naive random split (the before).** A plain unstratified 80/20 `train_test_split`. This is *not* what `scripts/03_train_model.py` does: that script holds out 20% of clients via `make_client_aware_split` and prints `Split strategy: client_holdout`. I build the naive row split here on purpose, as a lower bound, to measure how much client memorization an ungrouped split would pay out. Clients land on both sides. Answers: how good does the model look when it is allowed to re-describe pages it has already seen?
- **Row 2, client-grouped CV (Week 5's design).** Same five folds, same seed, so Week 5's committed numbers must reproduce. Adds ROC-AUC and Brier. Answers: does it rank pages for a client it never saw?
- **Row 3, one step forward in time.** Train on the March frame, test on an April frame built the same way (April features, May labels). Answers: does the ranking survive one month of drift?

```
TRAIN:  features Mar 01-31 | label Apr 01-30   <- Week-5 frame (rebuilt, receipt-checked)
TEST:   features Apr 01-30 | label May 01-30   <- same SQL, shifted windows
SEAL:   June                                    <- never read
```

- **Row 4, walk-forward with growing history.** Row 3 can fail for two very different reasons: the model only ever saw one month of conditions, or conditions genuinely moved. Rows 1-3 cannot tell those apart. So I go back and redo the out-of-time test at earlier cutoffs, letting the training set grow month by month, always testing into the same April. Answers: does more history fix it? Design is fixed before running; every origin gets reported, flattering or not.

Reporting rules: every metric sits next to its side's base rate; grouped results are fold means with spread, never single draws; definitions are identical across all four rows.

In [29]:
# Row 1 - BEFORE: naive random 80/20 row split. NOT the reference pipeline's design:
# scripts/03_train_model.py uses make_client_aware_split (client_holdout). Built here as a
# deliberate lower bound. No stratification, no grouping. Clients mix freely across
# the split - quantifying that mixing is the point of this row.
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss


def full_metrics(y_true, scores):
    return {
        "p20": precision_at_k(y_true, scores, 20),
        "p50": precision_at_k(y_true, scores, 50),
        "ndcg50": ndcg_at_k(y_true, scores, 50),
        "ap": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "brier": float(brier_score_loss(y_true, np.asarray(scores, dtype=float))),
    }


train_df, test_df = train_test_split(df_model, test_size=0.20, random_state=SEED)

tr_clients = set(train_df["client_hash_id"])
te_clients = set(test_df["client_hash_id"])
overlap_clients = tr_clients & te_clients

print(f"Random 80/20 row split: train={len(train_df):,} rows | test={len(test_df):,} rows")
print(f"Clients on BOTH sides: {len(overlap_clients)} of {len(tr_clients | te_clients)} "
      f"(train-only {len(tr_clients - te_clients)}, test-only {len(te_clients - tr_clients)})")
print(f"Base rates: train {train_df[TARGET].mean():.3f} | test {test_df[TARGET].mean():.3f}")

random_results = {}
base_test_scores = make_baseline_scores(test_df).to_numpy()
random_results["baseline"] = full_metrics(test_df[TARGET], base_test_scores)
for name, model in models.items():
    model.fit(train_df[FEATURES], train_df[TARGET])
    proba = model.predict_proba(test_df[FEATURES])[:, 1]
    random_results[name] = full_metrics(test_df[TARGET], proba)

print("\n=== Row 1: NAIVE RANDOM SPLIT (a lower bound; the reference script groups by client) ===")
random_table = pd.DataFrame(random_results).T
print(random_table.round(3).to_string())
print("\nThe grouped row below is the honest twin. The P@50 gap between them is")
print("how much client memorization the naive split quietly pays out.")

Random 80/20 row split: train=77,014 rows | test=19,254 rows
Clients on BOTH sides: 36 of 40 (train-only 4, test-only 0)
Base rates: train 0.509 | test 0.516

=== Row 1: NAIVE RANDOM SPLIT (a lower bound; the reference script groups by client) ===
                      p20   p50  ndcg50     ap  roc_auc  brier
baseline             0.65  0.48   0.537  0.522    0.499  0.420
logistic_regression  0.70  0.68   0.689  0.636    0.653  0.232
random_forest        1.00  1.00   1.000  0.732    0.734  0.209

The grouped row below is the honest twin. The P@50 gap between them is
how much client memorization the naive split quietly pays out.


In [30]:
# Row 2 - HONEST WITHIN-MONTH: Week 5's grouped design rerun identically (same seed,
# same sorted frame), extended with ROC-AUC and Brier.
from sklearn.model_selection import GroupShuffleSplit

groups = df_model["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=5, test_size=0.25, random_state=SEED)
fold_splits = list(splitter.split(df_model, y=df_model[TARGET], groups=groups))

comp_rows = []
for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    comp_rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "client_overlap": len(set(df_model["client_hash_id"].iloc[train_idx])
                              & set(df_model["client_hash_id"].iloc[test_idx])),
        "test_base_rate": round(float(df_model[TARGET].iloc[test_idx].mean()), 3),
    })
fold_composition = pd.DataFrame(comp_rows)
print(fold_composition.to_string(index=False))
assert fold_composition["client_overlap"].sum() == 0, "A client leaked between train and test sides"
print("Grouped-split check - no client on both sides of any fold: PASS")

cv_rows = []
for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    X_train, X_test = df_model[FEATURES].iloc[train_idx], df_model[FEATURES].iloc[test_idx]
    y_train, y_test = df_model[TARGET].iloc[train_idx], df_model[TARGET].iloc[test_idx]

    entry = {"fold": fold, "test_base_rate": float(y_test.mean())}
    entry["baseline"] = full_metrics(y_test, make_baseline_scores(df_model.iloc[test_idx]).to_numpy())
    for name, model in models.items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        entry[name] = full_metrics(y_test, proba)
    cv_rows.append(entry)
    print(f"fold {fold}: done")

cv_results = pd.DataFrame(cv_rows)

print("\n=== Row 2: CLIENT-GROUPED CV, 5 folds (Week-5 design reproduced + extended) ===")
summary_rows = []
for scorer in ["baseline", "logistic_regression", "random_forest"]:
    row = {"scorer": scorer}
    for m in ["p20", "p50", "ndcg50", "ap", "roc_auc", "brier"]:
        vals = cv_results[scorer].apply(lambda d: d[m])
        row[f"{m}_mean"] = float(vals.mean())
        row[f"{m}_sd"] = float(vals.std())
    summary_rows.append(row)
grouped_summary = pd.DataFrame(summary_rows).set_index("scorer")
print(grouped_summary.round(3).to_string())
print(f"\nMean test-fold base rate: {cv_results['test_base_rate'].mean():.3f}")

 fold  train_rows  test_rows  client_overlap  test_base_rate
    1       71678      24590               0           0.636
    2       71959      24309               0           0.421
    3       28283      67985               0           0.485
    4       70097      26171               0           0.408
    5       70797      25471               0           0.599
Grouped-split check - no client on both sides of any fold: PASS
fold 1: done
fold 2: done
fold 3: done
fold 4: done
fold 5: done

=== Row 2: CLIENT-GROUPED CV, 5 folds (Week-5 design reproduced + extended) ===
                     p20_mean  p20_sd  p50_mean  p50_sd  ndcg50_mean  ndcg50_sd  ap_mean  ap_sd  roc_auc_mean  roc_auc_sd  brier_mean  brier_sd
scorer                                                                                                                                         
baseline                 0.43   0.313     0.512   0.214        0.497      0.232    0.514  0.118         0.497       0.028       0.440   

In [31]:
# Row 3 prep: the April-cutoff frame. Features from month=2026-04, labels from
# month=2026-05. Same builder, shifted windows. June is never touched.
df_april, meta_april = build_frame("month=2026-04")
df_april_model = df_april.dropna(subset=FEATURES).copy()

print(f"\nApril frame rows: {len(df_april):,}")
print(f"Label distribution: {df_april['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {meta_april['cutoff']}")

# An out-of-time test compares frames one month apart, so the reader needs to know
# how much the population itself moved.
march_grains = set(zip(df_march["client_hash_id"], df_march["content_hash_id"]))
april_grains = set(zip(df_april["client_hash_id"], df_april["content_hash_id"]))
grain_overlap = len(march_grains & april_grains) / len(april_grains)

shift_table = pd.DataFrame([
    {"frame": "march (train)", **{k: meta_march[k] for k in ["rows", "base_rate", "clients", "largest_client_share"]}},
    {"frame": "april (test)", **{k: meta_april[k] for k in ["rows", "base_rate", "clients", "largest_client_share"]}},
])
print("\n=== Frame shift, March -> April ===")
print(shift_table.round(3).to_string(index=False))
print(f"Share of April-frame page-grains also present in the March frame: {grain_overlap:.1%}")

Feature month month=2026-04 | cutoff 2026-04-30 | label window 2026-05-01 .. 2026-05-30 | label partitions ['2026-05'] | bounds 2026-05-01 .. 2026-05-31

April frame rows: 99,279
Label distribution: 56.0% declining
Cutoff date: 2026-04-30

=== Frame shift, March -> April ===
        frame  rows  base_rate  clients  largest_client_share
march (train) 96268      0.511       40                 0.221
 april (test) 99279      0.560       44                 0.224
Share of April-frame page-grains also present in the March frame: 85.1%


In [32]:
# Row 3 - AFTER: fit once on ALL of March, score April. The scaler is fitted on March
# only; April is transformed, never fitted - the Pipeline boundary makes this honest.
oot_results = {}
oot_base_scores = make_baseline_scores(df_april_model).to_numpy()
oot_results["baseline"] = full_metrics(df_april_model[TARGET], oot_base_scores)
for name, model in models.items():
    model.fit(df_model[FEATURES], df_model[TARGET])
    proba = model.predict_proba(df_april_model[FEATURES])[:, 1]
    oot_results[name] = full_metrics(df_april_model[TARGET], proba)

print("=== Row 3: TIME-FORWARD (train March -> test April) ===")
oot_table = pd.DataFrame(oot_results).T
print(oot_table.round(3).to_string())
print(f"\nTest-side base rate (April frame): {df_april_model[TARGET].mean():.3f}"
      f"  <-- every number above reads against THIS")

=== Row 3: TIME-FORWARD (train March -> test April) ===
                      p20   p50  ndcg50     ap  roc_auc  brier
baseline             0.50  0.44   0.453  0.559    0.511  0.410
logistic_regression  0.55  0.54   0.521  0.626    0.606  0.240
random_forest        0.60  0.54   0.604  0.588    0.582  0.252

Test-side base rate (April frame): 0.560  <-- every number above reads against THIS


In [33]:
# Consolidated before/after: three splits x three scorers x the headline metrics.
# Top-K numbers are fold means (+/- sd) for the grouped row; single-fit numbers for the
# random and time rows. Base rates underneath, so nothing floats free.
scorer_names = ["baseline", "logistic_regression", "random_forest"]

table_rows = []
for scorer in scorer_names:
    r = random_results[scorer]
    g = grouped_summary.loc[scorer]
    o = oot_results[scorer]
    table_rows.append({
        "scorer": scorer,
        "P@50_random": r["p50"],
        "P@50_grouped_mean": g["p50_mean"],
        "P@50_grouped_sd": g["p50_sd"],
        "P@50_time_fwd": o["p50"],
        "AP_random": r["ap"],
        "AP_grouped_mean": g["ap_mean"],
        "AP_time_fwd": o["ap"],
        "ROC-AUC_random": r["roc_auc"],
        "ROC-AUC_grouped_mean": g["roc_auc_mean"],
        "ROC-AUC_time_fwd": o["roc_auc"],
        "Brier_random": r["brier"],
        "Brier_grouped_mean": g["brier_mean"],
        "Brier_time_fwd": o["brier"],
    })
before_after = pd.DataFrame(table_rows).set_index("scorer")
print("=== Three-row before/after (identical metrics, identical definitions) ===")
print(before_after.round(3).to_string())
print(f"\nBase rates: random-test {test_df[TARGET].mean():.3f} | "
      f"grouped folds mean {cv_results['test_base_rate'].mean():.3f} | "
      f"time-forward test {df_april_model[TARGET].mean():.3f}")
print("Random and time rows are single fits; the grouped row carries the spread.")

=== Three-row before/after (identical metrics, identical definitions) ===
                     P@50_random  P@50_grouped_mean  P@50_grouped_sd  P@50_time_fwd  AP_random  AP_grouped_mean  AP_time_fwd  ROC-AUC_random  ROC-AUC_grouped_mean  ROC-AUC_time_fwd  Brier_random  Brier_grouped_mean  Brier_time_fwd
scorer                                                                                                                                                                                                                                
baseline                    0.48              0.512            0.214           0.44      0.522            0.514        0.559           0.499                 0.497             0.511         0.420               0.440           0.410
logistic_regression         0.68              0.640            0.196           0.54      0.636            0.578        0.626           0.653                 0.608             0.606         0.232               0.257           0.240
ra

**Result.**

**The naive split looks perfect, and that is the problem.** Random forest reaches precision@50 of 1.000 on the random split against a grouped-fold mean of 0.740. The split report explains it: 36 of the 40 clients sit on both sides, so most test pages were already familiar from training. That perfect queue is memory, not forecasting. Logistic regression shows the same effect more quietly (0.680 random versus 0.640 grouped).

**One month forward, most of the lift dies.** Trained on March and scored on April, RF precision@50 falls to 0.540 while the April base rate is 0.560 - the top of the queue stops beating a coin flip. ROC-AUC slides from 0.606 to 0.582. Brier lands at 0.252, basically the base-rate predictor's 0.246, so the probabilities carry little usable information out of month. LR keeps somewhat more global order (average precision 0.626) but also drops to 0.54 at the head. Plain reading: what March taught the model about ranking the top slots did not survive the month.

**Client mix, corrected with this week's numbers.** The largest client holds 22.1% of frame rows. Week 5 wrote that fold 3 was dominated by one whale client, but that mixed up two things: fold 3's test draw holds 67,985 rows (about 70% of the frame) spread across ten clients. No single client owns that mass. This notebook corrects the record rather than repeating it.

Every P@50 above reads against its own side's base rate: 0.516 random test, 0.510 grouped folds, 0.560 April.

**Verdict:** Observed across five client-grouped folds in March 2026, both learned models ranked the decline-review queue at or above the hand rule everywhere they met it (means: rule 0.512, LR 0.640, RF 0.740, against a 0.510 fold base rate). Measured one month forward, the head-of-queue lift mostly disappeared (RF 0.54 against an April base of 0.56). As decision-support: this is a tool to refit monthly and use alongside human judgment. My one-line takeaway: the queue's output can stand behind its claim only for the month directly following its training data; use it accordingly, as a tool beside judgment, not as a replacement for it.

**Row 4 design, fixed before running.** The Row 3 collapse has two candidate causes and Rows 1-3 cannot tell them apart: the model only saw one month of conditions, or conditions genuinely moved. Row 4 tests the first cause directly.

The test month stays fixed on April, the same April as Row 3. Only the training history grows:

```
origin 1:  train (Nov feats -> Dec labels)                        -> test April
origin 2:  train + (Dec feats -> Jan labels)                      -> test April
origin 3:  train + (Jan feats -> Feb labels)                      -> test April
origin 4:  train + (Feb feats -> Mar labels)                      -> test April
           (= Row 3's training set, for a clean anchor)
```

Configs stay the inherited defaults, no tuning between origins, and every origin gets reported. If April precision climbs as history grows, the Row 3 gap was ours and history closes it. If it stays flat or falls, drift is real and monthly refitting is what the data demands. Either way, measurement decides, not preference.

Two disclosures before any numbers exist:

- Adjacent frames share most of the same pages (March and April overlapped 85% by grain). Expanding unions therefore contain correlated rows: row count grows faster than true information. Legal for training, but said out loud.
- Early months may be thin. GSC history depth differs by client, and some clients start later. The first cell below checks coverage and prints every frame's size and base rate before anything trains.

In [34]:
# Row 4 step 1: who has data this far back, and can the early frames hold weight?
print("=== Client GSC history start (quantiles) ===")
q = con.sql(f"""
    SELECT quantile_cont(TRY_CAST(gsc_data_start AS DATE), [0.1, 0.25, 0.5, 0.75, 0.9])
    FROM read_parquet('{REL}/dim_clients.parquet')
""").fetchone()[0]
for name, d in zip(["p10", "p25", "p50", "p75", "p90"], q):
    print(f"  {name}: {d}")

# Origins pre-committed in the markdown above: Nov 2025 onward, oldest first.
EARLY = [
    ("nov", "month=2025-11"),
    ("dec", "month=2025-12"),
    ("jan", "month=2026-01"),
    ("feb", "month=2026-02"),
]
early_frames = {}
early_meta = {}
for tag, fm in EARLY:
    f, m = build_frame(fm)
    early_frames[tag] = f.dropna(subset=FEATURES).copy()
    early_meta[tag] = m
    print(f"  frame {tag}: rows={len(f):,} | base={m['base_rate']:.3f} "
          f"| clients={m['clients']} | largest_client={m['largest_client_share']:.1%}")

sizes = {t: len(v) for t, v in early_frames.items()}
print(f"\nAll four early frames built. Smallest: {min(sizes.values()):,} rows.")
assert min(sizes.values()) > 1000, "An early frame is too thin to train on - reconsider origins"

=== Client GSC history start (quantiles) ===
  p10: 2025-06-21 00:00:00
  p25: 2025-09-24 00:00:00
  p50: 2025-11-05 00:00:00
  p75: 2026-02-19 00:00:00
  p90: 2026-04-18 19:12:00
Feature month month=2025-11 | cutoff 2025-11-30 | label window 2025-12-01 .. 2025-12-30 | label partitions ['2025-12'] | bounds 2025-12-01 .. 2025-12-31
  frame nov: rows=44,165 | base=0.238 | clients=26 | largest_client=26.1%
Feature month month=2025-12 | cutoff 2025-12-31 | label window 2026-01-01 .. 2026-01-30 | label partitions ['2026-01'] | bounds 2026-01-01 .. 2026-01-31
  frame dec: rows=51,415 | base=0.236 | clients=27 | largest_client=25.2%
Feature month month=2026-01 | cutoff 2026-01-31 | label window 2026-02-01 .. 2026-03-02 | label partitions ['2026-02', '2026-03'] | bounds 2026-02-01 .. 2026-03-31
  frame jan: rows=60,869 | base=0.213 | clients=27 | largest_client=24.6%
Feature month month=2026-02 | cutoff 2026-02-28 | label window 2026-03-01 .. 2026-03-30 | label partitions ['2026-03'] | bounds 

In [35]:
# Row 4 step 2: expanding-history walk-forward into the SAME April test as Row 3.
# Only the amount of training history changes between origins. Nothing is tuned;
# every origin is reported.
from sklearn.base import clone

ORDER = ["nov", "dec", "jan", "feb"]
TEST_FRAME = df_april_model
y_test_wf = TEST_FRAME[TARGET]
wf_base_scores = make_baseline_scores(TEST_FRAME).to_numpy()

wf_rows = []
wf_records = []  # json-safe copy for the receipts file
pool = []
for tag in ORDER:
    pool.append(early_frames[tag])
    train = pd.concat(pool, ignore_index=True).sort_values(
        ["client_hash_id", "content_hash_id"]).reset_index(drop=True)

    entry = {
        "history_through": tag,
        "train_frames": len(pool),
        "train_rows": len(train),
        "train_clients": int(train["client_hash_id"].nunique()),
        "test_base_rate": float(y_test_wf.mean()),
    }
    metrics_entry = {}
    entry["baseline"] = full_metrics(y_test_wf, wf_base_scores)
    for name, mdl in models.items():
        fitted_wf = clone(mdl).fit(train[FEATURES], train[TARGET])
        proba = fitted_wf.predict_proba(TEST_FRAME[FEATURES])[:, 1]
        entry[name] = full_metrics(y_test_wf, proba)
        metrics_entry[name] = entry[name]
    wf_rows.append(entry)
    wf_records.append({
        "history_through": tag,
        "train_frames": entry["train_frames"],
        "train_rows": int(entry["train_rows"]),
        "train_clients": entry["train_clients"],
        "test_base_rate": float(entry["test_base_rate"]),
        "metrics": {"baseline": {k: float(v) for k, v in entry["baseline"].items()},
                    **{n: {k: float(v) for k, v in d.items()} for n, d in metrics_entry.items()}},
    })
    print(f"origin {entry['train_frames']} (history through {tag}): done")

wf_df = pd.DataFrame([
    {
        "history_through": e["history_through"],
        "train_frames": e["train_frames"],
        "train_rows": e["train_rows"],
        **{f"{s}_{m}": v for s in ("baseline", "logistic_regression", "random_forest")
           for m, v in e[s].items()},
    }
    for e in wf_rows
])

print("\n=== Row 4: EXPANDING HISTORY -> SAME APRIL TEST (base rate "
      f"{y_test_wf.mean():.3f}) ===")
show_cols = ["history_through", "train_frames", "train_rows",
             "baseline_p50", "baseline_ap",
             "logistic_regression_p50", "logistic_regression_ap", "logistic_regression_brier",
             "random_forest_p50", "random_forest_ap", "random_forest_brier"]
print(wf_df[show_cols].round(3).to_string(index=False))

origin 1 (history through nov): done
origin 2 (history through dec): done
origin 3 (history through jan): done
origin 4 (history through feb): done

=== Row 4: EXPANDING HISTORY -> SAME APRIL TEST (base rate 0.560) ===
history_through  train_frames  train_rows  baseline_p50  baseline_ap  logistic_regression_p50  logistic_regression_ap  logistic_regression_brier  random_forest_p50  random_forest_ap  random_forest_brier
            nov             1       44164          0.44        0.559                     0.54                   0.535                      0.319               0.26             0.561                0.275
            dec             2       95579          0.44        0.559                     0.74                   0.564                      0.336               0.50             0.633                0.248
            jan             3      156445          0.44        0.559                     0.56                   0.571                      0.343               0.46         

**Walk-forward finding:** Adding training frames helps only briefly, then stops helping. Observed across the four origins, both models peak with two frames of history (through December): LR precision@50 jumps from 0.54 to 0.74, RF from 0.26 to 0.50. From there the head slides back down - LR to 0.56 then 0.50, RF to 0.46 then 0.48 - while average precision stays roughly flat for both. The hand rule reads 0.44 on every row, which is expected: its score is a fixed formula that never touches training data, so the identical baseline also works as a check that the test side never moved. Verdict: the not-enough-history explanation for the Row-3 drop is rejected as the main cause. One extra month of conditions helped, but no amount of past months restores March-level head precision against April (best single out-of-time draw 0.74, unstable after, against a 0.560 base rate and a 0.740 grouped within-March mean). Decision-support conclusion: with these five features the queue is a one-month instrument - refit it monthly and let each queue speak only for the month directly after its training data.

## 3. Leakage audit

Week 3 hunted leaks in the data contract. This audit reruns the hunt on the final feature set, plus two places Week 3 never looked: the population definition and the April frame. Each item maps to the skill's attack checklist, and each one is either proven or measured below.

- **Timeline drawn.** Features end strictly before the label window opens, asserted for every frame built here.
- **No label-derived features.** The model sees exactly the five contracted columns; trend and future fields never enter. And to prove the audit itself works, I plant the known leak and watch the score jump. An audit that cannot catch a planted leak cannot clear an honest one.
- **No product flags.** The Week-4 rule score stays a baseline to beat, never an input.
- **Population checked for outcome-window information.** The INNER JOIN keeps only pages with at least 14 days of GSC coverage in the outcome month. Week 3 left that implicit. Here it gets counted.
- **Both honest axes covered.** Grouped splits for unseen clients, walk-forward for unseen time. Base rates print next to every metric. Metrics are out-of-fold throughout. June stays sealed. The numbers land in a committed receipts file.

In [36]:
# Structural checks: the five-feature contract, forbidden names, timeline separation.
print("=== Contract: the model sees EXACTLY the five contracted features ===")
expected = {
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
}
assert set(FEATURES) == expected and len(FEATURES) == 5, "Feature contract drifted"
assert TARGET not in FEATURES, "Target leaked into features"
print("PASS:", sorted(FEATURES))

print("\n=== Forbidden-name scan on the modeling matrix ===")
forbidden_markers = ("trend", "future", "label", "score", "direction")
present_forbidden = [c for c in FEATURES if any(m in c.lower() for m in forbidden_markers)]
assert not present_forbidden, f"Forbidden column reached the feature matrix: {present_forbidden}"

helper_cols_present = [c for c in df_model.columns
                       if any(m in c.lower() for m in ("trend", "future"))]
print("No forbidden names among FEATURES: PASS")
print(f"In the wider frame but never in FEATURES (label inputs / injection test only): "
      f"{helper_cols_present}")

print("\n=== Timeline: feature window ends strictly before the label window, every frame ===")
all_metas = [("march", meta_march), ("april", meta_april)] + \
            [(t, early_meta[t]) for t in ["nov", "dec", "jan", "feb"]]
for tag, meta in all_metas:
    cutoff = date.fromisoformat(meta["cutoff"])
    ls, le = (date.fromisoformat(x) for x in meta["label_window"])
    assert cutoff < ls <= le
    print(f"PASS {tag}: features ..{meta['cutoff']} | labels {ls}..{le}")

=== Contract: the model sees EXACTLY the five contracted features ===
PASS: ['content_age_days', 'log_recent30_impressions', 'recent30_active_days', 'recent30_avg_position', 'recent30_ctr_pct']

=== Forbidden-name scan on the modeling matrix ===
No forbidden names among FEATURES: PASS
In the wider frame but never in FEATURES (label inputs / injection test only): ['future30_impressions']

=== Timeline: feature window ends strictly before the label window, every frame ===
PASS march: features ..2026-03-31 | labels 2026-04-01..2026-04-30
PASS april: features ..2026-04-30 | labels 2026-05-01..2026-05-30
PASS nov: features ..2025-11-30 | labels 2025-12-01..2025-12-30
PASS dec: features ..2025-12-31 | labels 2026-01-01..2026-01-30
PASS jan: features ..2026-01-31 | labels 2026-02-01..2026-03-02
PASS feb: features ..2026-02-28 | labels 2026-03-01..2026-03-30


In [37]:
# Harness sensitivity test: plant the known-leaky column (the label's own input) and
# watch the score confess. If a planted leak does NOT move the score toward 1.0,
# this audit's clean verdicts are worthless.
from sklearn.base import clone

# Keep the FULL fold frames; narrow to FEATURES only when fitting the honest model.
# The leaky column lives beside FEATURES in the wide frame - slicing first broke this
# test the first time it ran.
train_idx, test_idx = fold_splits[0]
train_wide, test_wide = df_model.iloc[train_idx], df_model.iloc[test_idx]
X_train, X_test = train_wide[FEATURES], test_wide[FEATURES]
y_train, y_test = train_wide[TARGET], test_wide[TARGET]

clean = clone(models["random_forest"]).fit(X_train, y_train)
clean_ap = float(average_precision_score(y_test, clean.predict_proba(X_test)[:, 1]))

leaky_features = FEATURES + ["future30_impressions"]
leaky = clone(models["random_forest"]).fit(train_wide[leaky_features], y_train)
leaky_ap = float(average_precision_score(y_test, leaky.predict_proba(test_wide[leaky_features])[:, 1]))

print(f"Honest 5-feature AP (grouped fold 1):        {clean_ap:.3f}")
print(f"With future30_impressions injected (6 feats): {leaky_ap:.3f}")
print(f"Week-3 deliberate-leak reference:             ~0.998")
delta = leaky_ap - clean_ap
print(f"Jump: +{delta:.3f}")
assert delta > 0.15, "Harness failed to detect a planted leak - audit tooling is broken"
print("\nHarness sensitivity: PASS - the audit can detect a leak when one exists.")
print("Column removed again; everything downstream runs on the honest five.")

Honest 5-feature AP (grouped fold 1):        0.779
With future30_impressions injected (6 feats): 0.992
Week-3 deliberate-leak reference:             ~0.998
Jump: +0.212

Harness sensitivity: PASS - the audit can detect a leak when one exists.
Column removed again; everything downstream runs on the honest five.


In [38]:
# Population-selection audit: how many candidate pages does the INNER-JOIN-future
# design exclude, and why? A page enters a frame only if its recent side passes the
# floors AND the outcome month shows >= 14 days of GSC coverage.
survivorship_sql_tpl = """
    WITH recent_pool AS (
        SELECT client_hash_id, content_hash_id
        FROM {feat}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING COUNT(DISTINCT report_date) >= 14
           AND SUM(gsc_impressions) >= 100
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               COUNT(DISTINCT report_date) AS outcome_days
        FROM {fut}
        WHERE report_date BETWEEN DATE '{ls}' AND DATE '{le}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*) AS pool_candidates,
        COUNT(*) FILTER (WHERE o.outcome_days IS NOT NULL AND o.outcome_days >= 14) AS labeled,
        COUNT(*) FILTER (WHERE o.outcome_days IS NULL) AS dropped_absent_outcome,
        COUNT(*) FILTER (WHERE o.outcome_days IS NOT NULL AND o.outcome_days < 14) AS dropped_thin_outcome
    FROM recent_pool p
    LEFT JOIN outcome o USING (client_hash_id, content_hash_id)
"""


def survivorship(meta):
    fut_paths = ", ".join(
        f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'"
        for m in meta["future_partitions"]
    )
    return con.sql(survivorship_sql_tpl.format(
        feat=f"read_parquet('{REL}/fact_content_daily_performance/{meta['feature_month']}/*.parquet')",
        fut=f"read_parquet([{fut_paths}])",
        ls=meta["label_window"][0], le=meta["label_window"][1],
    )).df().iloc[0]


surv_march = survivorship(meta_march)
surv_april = survivorship(meta_april)

print("=== Population selection: who never makes it into the frame? ===")
surv_table = pd.DataFrame([
    {"frame": "march->april",
     "candidates_meeting_recent_floors": int(surv_march["pool_candidates"]),
     "labeled_in_frame": int(surv_march["labeled"]),
     "dropped_no_outcome_rows": int(surv_march["dropped_absent_outcome"]),
     "dropped_outcome_lt14d": int(surv_march["dropped_thin_outcome"])},
    {"frame": "april->may",
     "candidates_meeting_recent_floors": int(surv_april["pool_candidates"]),
     "labeled_in_frame": int(surv_april["labeled"]),
     "dropped_no_outcome_rows": int(surv_april["dropped_absent_outcome"]),
     "dropped_outcome_lt14d": int(surv_april["dropped_thin_outcome"])},
])
surv_table["dropped_pct_of_candidates"] = (
    100.0 * (surv_table["candidates_meeting_recent_floors"] - surv_table["labeled_in_frame"])
    / surv_table["candidates_meeting_recent_floors"]
)
print(surv_table.to_string(index=False))

# Internal consistency: the survivorship query must reproduce the built frames exactly.
assert int(surv_march["labeled"]) == len(df_march), "Survivorship query disagrees with built March frame"
assert int(surv_april["labeled"]) == len(df_april), "Survivorship query disagrees with built April frame"
print("\nConsistency: survivorship counts equal the built frames exactly: PASS")

# ---- receipts ----
full_rule_scores = make_baseline_scores(df_model)
band_mask = full_rule_scores == full_rule_scores.max()

receipt = {
    "notebook": "w06_validation_audit",
    "frames": {"march": meta_march, "april": meta_april},
    "random_split": {
        "shared_clients_both_sides": int(len(overlap_clients)),
        "test_base_rate": float(test_df[TARGET].mean()),
        "metrics": {k: {m: float(v) for m, v in d.items()} for k, d in random_results.items()},
    },
    "grouped_cv": {
        "per_fold": [
            {"fold": int(r["fold"]),
             "test_base_rate": float(r["test_base_rate"]),
             **{s: {m: float(v) for m, v in r[s].items()} for s in scorer_names}}
            for _, r in cv_results.iterrows()
        ],
        "summary": {s: {m: float(v) for m, v in grouped_summary.loc[s].items()} for s in scorer_names},
    },
    "time_forward": {
        "test_base_rate": float(df_april_model[TARGET].mean()),
        "metrics": {k: {m: float(v) for m, v in d.items()} for k, d in oot_results.items()},
    },
    "walk_forward": {"test_frame": "april", "origins": wf_records},
    "leak_injection": {"honest_ap": clean_ap, "injected_ap": leaky_ap},
    "survivorship": {
        "march_to_april": {k: int(v) for k, v in surv_march.items()},
        "april_to_may": {k: int(v) for k, v in surv_april.items()},
    },
    "rule_tie_band": {"n": int(band_mask.sum()), "decline_rate": float(df_model.loc[band_mask, TARGET].mean())},
    "seed": SEED,
    "library_versions": {"pandas": pd.__version__, "numpy": np.__version__,
                         "scikit-learn": sklearn.__version__, "duckdb": duckdb.__version__},
}
out_path = Path("../../work/outputs/validation_audit_metrics.json")
out_path.write_text(json.dumps(receipt, indent=2))
print(f"\nReceipts written: {out_path}")

=== Population selection: who never makes it into the frame? ===
       frame  candidates_meeting_recent_floors  labeled_in_frame  dropped_no_outcome_rows  dropped_outcome_lt14d  dropped_pct_of_candidates
march->april                             99000             96268                      412                   2320                   2.759596
  april->may                            102943             99279                      506                   3158                   3.559251

Consistency: survivorship counts equal the built frames exactly: PASS

Receipts written: ..\..\work\outputs\validation_audit_metrics.json


**Leakage-audit verdict, item by item:**

- **Timeline drawn: PASS.** Asserted at build time for all six frames; one day separates each feature window's end from its label window's start.
- **Feature contract: PASS.** Exactly the five contracted features reach the model. `future30_impressions` exists in the wider frame only to define the label and power the injection test below.
- **Harness sensitivity: PASS.** Planting that column lifts AP from **0.779 to 0.992** (+0.212), right next to the Week-3 deliberate-leak mark of about 0.998. The audit can catch a leak. Column removed; every result in this notebook uses the honest five.
- **No product flags: PASS.** The rule score is computed for evaluation only and asserted absent from the features.
- **Population selection: MEASURED, NOW DISCLOSED.** March keeps 96,268 of 99,000 recent-side candidates (**2.76% dropped**: 412 with no April rows, 2,320 under 14 covered days). April keeps 99,279 of 102,943 (**3.56% dropped**). Small shares, but the bias points one way: pages whose tracking died get excluded, and those skew toward pages going quiet. Everything in this notebook describes pages measurable next month, not all pages. Said, not hidden.
- **Splits, base rates, out-of-fold metrics, sealed June, receipts: PASS.**

Net effect on my claims: the Week-5 numbers stand as within-March, client-honest, next-month-measurable. Nothing more. Section 4 rewrites the sentence that drifted past that boundary.

## 4. Claim rewrite

The boldest sentence in my Week-5 notebook, quoted exactly:

> "Vs the baseline: neither model ever loses a fold at precision@50 - LR wins all five outright (5/5); RF wins four and ties one (fold 3, 0.40 vs 0.40) - the lift is consistent across client draws, not a lucky fold."

Why it goes further than the evidence:

1. "Never loses" sounds like a law. The proof is five reshuffles of one month's rows. Those folds share all their data, so they act like one experiment told five times, not five experiments.
2. The sentence drops its scope: March 2026, about 40 clients, five features, untuned defaults. Without the scope tags it claims more than it tested.
3. It had not faced time. Row 3 ran that test: head-of-queue precision fell from a 0.740 grouped mean to 0.540 against an April base of 0.560.

Rewritten with safe words:

> "Observed across five client-grouped folds within a single development month (March 2026, ~40 clients, five contracted features): both learned models scored at or above the hand rule's precision@50 in every fold - logistic regression won 5/5 outright, random forest 4/5 with one tie. Measured fold means: rule 0.512, LR 0.640, RF 0.740, against a 0.510 mean fold base rate. This is directional evidence that learned scoring orders the refresh-review queue better than the tie-bound rule under these conditions, offered as decision-support for editors. It does not establish performance across months: measured out-of-time on the April frame, head-of-queue precision fell to 0.54 against a 0.56 base rate - one month, one portfolio, directional only."

One more fix while I am here. Week 5 also said the hand rule "performs at chance on average." Sharper: the rule's *top-50 draw* performs at chance. Its signal is real - the score-1.00 band of 24,775 pages declines at 0.532 against a 0.511 frame base - but the rule cannot rank inside its own tie band. That is signal without resolution, not absence of signal. The two diagnoses lead to different next steps.

House rules for my claims from here on: **observed** means it happened in my tests, no universe implied. **Measured** means reproducible numbers with spread attached. **Directional** means a tendency, never a promise. **Decision-support** means it helps a person decide, never acts alone. The cell below prints every number the rewrites cite, so the prose carries nothing unverifiable.

In [39]:
# Every quantity cited by the Section-4 rewrite, printed where the reader can check it.
_rule_scores_full = make_baseline_scores(df_model)
_band_mask = _rule_scores_full == _rule_scores_full.max()
band_n = int(_band_mask.sum())
band_rate = float(df_model.loc[_band_mask, TARGET].mean())
whale_share = float(df_model["client_hash_id"].value_counts(normalize=True).iloc[0])

g = grouped_summary.loc["random_forest"]
print("Quantities backing the rewritten claim:")
print(f"  Grouped P@50, RF: mean {g['p50_mean']:.3f} (sd {g['p50_sd']:.3f}) over 5 client draws, ONE month")
print(f"  Largest single client share of frame rows: {whale_share:.1%}")
print(f"  Rule tie band: n={band_n:,}, decline rate {band_rate:.3f} (frame base {meta_march['base_rate']:.3f})")
print(f"  Random-split P@50 RF: {random_results['random_forest']['p50']:.3f} "
      f"(naive-split inflation vs grouped mean: "
      f"{random_results['random_forest']['p50'] - g['p50_mean']:+.3f})")
print(f"  Time-forward P@50 RF: {oot_results['random_forest']['p50']:.3f} "
      f"(vs grouped mean: {oot_results['random_forest']['p50'] - g['p50_mean']:+.3f})")
if wf_records:
    last = wf_records[-1]["metrics"]["random_forest"]["p50"]
    first = wf_records[0]["metrics"]["random_forest"]["p50"]
    print(f"  Walk-forward P@50 RF, thinnest -> fullest history: {first:.3f} -> {last:.3f}")

Quantities backing the rewritten claim:
  Grouped P@50, RF: mean 0.740 (sd 0.295) over 5 client draws, ONE month
  Largest single client share of frame rows: 22.1%
  Rule tie band: n=24,775, decline rate 0.532 (frame base 0.511)
  Random-split P@50 RF: 1.000 (naive-split inflation vs grouped mean: +0.260)
  Time-forward P@50 RF: 0.540 (vs grouped mean: -0.200)
  Walk-forward P@50 RF, thinnest -> fullest history: 0.260 -> 0.480


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.